# Tutorial 2: Run Retrieval and Reranking Evaluation on CoREB

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hq-bench/coreb/blob/main/notebooks/02_run_evaluation.ipynb)

CoREB evaluates code search in two stages:
1. **Retrieval** — an embedding model encodes queries and documents, then top-k candidates are retrieved via cosine similarity
2. **Reranking** — a cross-encoder reranker rescores the top-k candidates for improved precision

In this notebook, you will learn how to:
1. Load CoREB data and convert it to evaluation format
2. Run **Stage 1: Dense Retrieval** with a HuggingFace or Gemini embedding model
3. Run **Stage 2: Reranking** with [`coreb-code-reranker`](https://huggingface.co/hq-bench/coreb-code-reranker)
4. Evaluate both stages with graded-relevance metrics (nDCG, Recall, MAP, MRR)

**GPU recommended** — go to Runtime > Change runtime type > T4 GPU.

**Reference:** Xue et al., *Beyond Retrieval: A Multitask Benchmark and Model for Code Search*, 2025. [arXiv:2605.04615](https://arxiv.org/abs/2605.04615)

## 1. Installation

In [ ]:
!pip install -q "coreb[hf]"

## 2. Load Data from HuggingFace

We load the **v202603 (test)** split and convert it into the dict format expected by the evaluator.

In [ ]:
from datasets import load_dataset
from coreb_runner.benchmark import (
    convert_corpus_to_coir_format,
    convert_queries_to_coir_format,
    convert_qrels_to_coir_format,
)

SPLIT = "release_v2603"

# Load code corpus and T2C task as an example
code_corpus_hf = load_dataset("hq-bench/coreb", "code_corpus",       split=SPLIT)
t2c_queries_hf = load_dataset("hq-bench/coreb", "text2code_queries", split=SPLIT)
t2c_qrels_hf   = load_dataset("hq-bench/coreb", "text2code_qrels",   split=SPLIT)

# Convert to evaluation format
corpus  = convert_corpus_to_coir_format(list(code_corpus_hf))
queries = convert_queries_to_coir_format(list(t2c_queries_hf))
qrels   = convert_qrels_to_coir_format(list(t2c_qrels_hf))

print(f"Corpus:  {len(corpus):,} documents")
print(f"Queries: {len(queries):,}")
print(f"Qrels:   {len(qrels):,} query groups")

## 3. Stage 1: Dense Retrieval

### 3.1 Initialize an Embedding Model

CoREB supports two model backends:
- **HuggingFace** (`model_type="huggingface"`): any Transformers-compatible encoder
- **Gemini** (`model_type="gemini"`): Google's embedding API (requires `GEMINI_API_KEY`)

Here we use a HuggingFace model as an example.

In [ ]:
from coreb_runner.benchmark import create_model_wrapper

# Choose a model — any HuggingFace encoder works.
# Smaller models are faster for demonstration:
MODEL_NAME = "jinaai/jina-embeddings-v3"

model = create_model_wrapper(MODEL_NAME, model_type="huggingface")
print(f"Model loaded: {MODEL_NAME}")

### 3.2 Run Retrieval

`DenseRetrievalExactSearch` encodes all queries and corpus documents, then computes brute-force cosine similarity to find top-k results.

In [ ]:
from coreb_runner.benchmark import DenseRetrievalExactSearch, EvaluateRetrieval

K_VALUES = [1, 3, 5, 10]

# Build retriever and evaluator
retriever = DenseRetrievalExactSearch(model, batch_size=64)
evaluator = EvaluateRetrieval(retriever, k_values=K_VALUES)

# Run retrieval
results = evaluator.retrieve(corpus, queries)

print(f"Retrieved results for {len(results):,} queries")

# Peek at top-3 results for the first query
first_qid = next(iter(results))
top3 = sorted(results[first_qid].items(), key=lambda x: x[1], reverse=True)[:3]
print(f"\nTop-3 for query '{first_qid}':")
for doc_id, score in top3:
    print(f"  {doc_id}: {score:.4f}")

### 3.3 Evaluate Retrieval Results

CoREB uses `relevance_level=2`: only true positives (rel>=2) count as relevant for binary metrics (Recall, MAP, Precision). Hard negatives (rel=1) penalize nDCG by occupying top ranks with zero gain.

In [ ]:
ndcg, _map, recall, precision = EvaluateRetrieval.evaluate(
    qrels, results, K_VALUES
)

print("=" * 40)
print(f"  Model: {MODEL_NAME}")
print(f"  Task:  Text-to-Code (T2C)")
print("=" * 40)
for k in K_VALUES:
    print(f"  nDCG@{k:<3d}  {ndcg[f'NDCG@{k}']:.4f}")
print("-" * 40)
for k in K_VALUES:
    print(f"  Recall@{k:<3d}{recall[f'Recall@{k}']:.4f}")
print("-" * 40)
for k in K_VALUES:
    print(f"  MAP@{k:<3d}   {_map[f'MAP@{k}']:.4f}")
print("-" * 40)
for k in K_VALUES:
    print(f"  P@{k:<3d}     {precision[f'P@{k}']:.4f}")

### 3.4 Custom Metrics: MRR and Top-k Accuracy

In [ ]:
mrr = EvaluateRetrieval.evaluate_custom(qrels, results, K_VALUES, metric="mrr")
acc = EvaluateRetrieval.evaluate_custom(qrels, results, K_VALUES, metric="accuracy")

print("MRR (only rel>=2 count as hits):")
for k in K_VALUES:
    print(f"  MRR@{k}: {mrr[f'MRR@{k}']:.4f}")

print("\nTop-k Accuracy:")
for k in K_VALUES:
    print(f"  Accuracy@{k}: {acc[f'Accuracy@{k}']:.4f}")

## 4. Stage 2: Reranking with CoREB-Reranker

After retrieval, a **cross-encoder reranker** rescores the top-k candidates for improved precision. [`coreb-code-reranker`](https://huggingface.co/hq-bench/coreb-code-reranker) is fine-tuned from Qwen3-Reranker-4B via LoRA and is the first reranker to achieve consistent gains across all three code search tasks.

The reranker takes a (query, document) pair and outputs a relevance score by comparing the logits of "yes" vs "no" tokens.

### 4.1 Load the Reranker

In [ ]:
from enum import Enum
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

class Task(Enum):
    TEXT_TO_CODE = "Given a natural language programming task, retrieve code that correctly solves or implements the task."
    CODE_TO_CODE = "Given a code snippet, retrieve code that is semantically equivalent or solves the same task."
    CODE_TO_TEXT = "Given a code snippet, retrieve the natural language description or problem statement that best matches the code."

RERANKER_ID = "hq-bench/coreb-code-reranker"

reranker_tokenizer = AutoTokenizer.from_pretrained(RERANKER_ID, trust_remote_code=True)
reranker_model = AutoModelForCausalLM.from_pretrained(
    RERANKER_ID, torch_dtype=torch.bfloat16, trust_remote_code=True
)
reranker_model.eval()

device = "cuda" if torch.cuda.is_available() else "cpu"
reranker_model.to(device)

# Prompt template for the reranker
PREFIX = '<|im_start|>system\nJudge whether the Document meets the requirements based on the Query and the Instruct provided. Note that the answer can only be "yes" or "no".<|im_end|>\n<|im_start|>user\n'
SUFFIX = "<|im_end|>\n<|im_start|>assistant\n"
yes_id = reranker_tokenizer.convert_tokens_to_ids("yes")
no_id  = reranker_tokenizer.convert_tokens_to_ids("no")

def reranker_score(query: str, document: str, task: Task) -> float:
    """Score a (query, document) pair using the cross-encoder reranker."""
    prompt = f"{PREFIX}<Instruct>: {task.value}\n<Query>: {query}\n<Document>: {document}{SUFFIX}"
    inputs = reranker_tokenizer(prompt, return_tensors="pt", truncation=True, max_length=4096).to(device)
    with torch.no_grad():
        logits = reranker_model(**inputs).logits[0, -1, :]
    return (logits[yes_id] - logits[no_id]).item()

print(f"Reranker loaded: {RERANKER_ID} on {device}")

### 4.2 Rerank Retrieval Results and Evaluate

We take the top-k candidates from Stage 1 and rescore them with the reranker, then re-evaluate.

## 5. Evaluate All Three Tasks

Use `CoREBEvaluation` to run retrieval and evaluation across multiple tasks in one go.

## 7. Evaluate All Three Tasks

Use `CoREBEvaluation` to run retrieval and evaluation across multiple tasks in one go.

## 6. Using the Gemini Embedding API (Optional)

To use Google's Gemini embedding models instead, set your API key and switch the model type.

## 8. Using the Gemini Embedding API (Optional)

To use Google's Gemini embedding models instead, set your API key and switch the model type.

In [ ]:
# Uncomment to use Gemini:

# import os
# os.environ["GEMINI_API_KEY"] = "your-api-key-here"
#
# gemini_model = create_model_wrapper(
#     "gemini-embedding-2-preview",
#     model_type="gemini",
#     batch_size=100,
# )
#
# retriever = DenseRetrievalExactSearch(gemini_model, batch_size=100)
# evaluator = EvaluateRetrieval(retriever, k_values=[1, 3, 5, 10])
# results = evaluator.retrieve(corpus, queries)
# ndcg, _map, recall, precision = evaluator.evaluate(qrels, results, [1, 3, 5, 10])
# print(f"Gemini nDCG@10: {ndcg['NDCG@10']:.4f}")

## Citation

If you use CoREB in your research, please cite:

```bibtex
@article{xue2025coreb,
  title   = {Beyond Retrieval: A Multitask Benchmark and Model for Code Search},
  author  = {Xue, Siqiao and Liao, Zihan and Qin, Jin and Zhang, Ziyin and Mu, Yixiang and Zhou, Fan and Yu, Hang},
  journal = {arXiv preprint arXiv:2605.04615},
  year    = {2025},
  url     = {https://arxiv.org/abs/2605.04615}
}
```